# 📊 Students Performance in Exams — Full BI & EDA Report
### Senior Sales/BI Data Analyst Deliverable — Colab-style Notebook

**Dataset:** [Students Performance in Exams](https://www.kaggle.com/datasets/spscientist/students-performance-in-exams) (Kaggle, 1000 rows × 8 columns)

**Scope of this notebook:**
1. Data Quality Report
2. Overview (Categorical & Numerical statistics, Outliers)
3. Important Business Questions (answered strictly from data)
4. EDA — Univariate, Bivariate, Multivariate
5. Performance Increase & Drop analysis (Why / When / Where)
6. Deep-dive study of student performance (risk & top segments)
7. Data Cleaning Recommendations (fill-logic, nothing removed)
8. Group Insight Reports: Boys vs Girls · Subject Comparison · Parents' Education · Overall Performance
9. Correlation vs Causation note

> **Ground rule followed throughout:** every statement below is derived directly from the dataset. No external assumption, no value is removed, and the original raw data is never modified — all new analysis lives in new sections/variables (the "new worksheets" of this notebook).


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/) 

**How to run this in Google Colab:**
1. Go to `File > Upload notebook` in Colab and upload this `.ipynb` file (or open it directly from Google Drive/GitHub if you host it there).
2. Run the **Setup & Load Data** cell below — it automatically downloads `StudentsPerformance.csv` from GitHub, so no manual upload of the CSV is required. If you'd rather use your own copy of the file, see the commented-out upload option in that cell.
3. Then choose `Runtime > Run all`.


## 0. Setup & Load Data
We load the raw CSV once, keep an **untouched copy** (`df_raw`) exactly as downloaded, and do all analysis on a working copy (`df`). No row or column from the source file is ever deleted.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats
import os, urllib.request

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# ---- Load data: works in Google Colab or locally ----
CSV_PATH = "/kaggle/input/datasets/spscientist/students-performance-in-exams/StudentsPerformance.csv"

df_raw = pd.read_csv(CSV_PATH)

df_raw.columns = [c.strip() for c in df_raw.columns]

print(df_raw.shape)
df_raw.head()# ---- Working copy for all analysis / new columns (raw data is never modified) ----
df = df_raw.copy()

print('Shape:', df.shape)
df.head()


In [ ]:
df.info()


## 1. Data Quality Report

We check: missing values, duplicates, invalid ranges, inconsistent text formatting, outliers and logical errors. This dataset has **no date/time column**, so no date-validity checks apply.


In [ ]:
dq = pd.DataFrame(index=df.columns)
dq['dtype'] = df.dtypes.astype(str)
dq['n_missing'] = df.isnull().sum()
dq['%_missing'] = (df.isnull().mean()*100).round(2)
dq['n_unique'] = df.nunique()
dq


In [ ]:
# Duplicate rows (full-row duplicates)
n_dupes = df.duplicated().sum()
print('Fully duplicated rows:', n_dupes)

# Duplicate check ignoring the 3 score columns (same profile, different scores is fine;
# identical profile AND identical scores would be suspicious)
n_dupes_profile_and_scores = df.duplicated(subset=df.columns.tolist()).sum()
print('Duplicate rows on all 8 fields:', n_dupes_profile_and_scores)


In [ ]:
# Categorical formatting consistency: check for stray whitespace / case variants
cat_cols = ['gender','race/ethnicity','parental level of education','lunch','test preparation course']
for c in cat_cols:
    print(c, '->', sorted(df[c].unique().tolist()))


In [ ]:
# Numeric range validity: scores must be within [0, 100]
score_cols = ['math score','reading score','writing score']
for c in score_cols:
    invalid = df[(df[c] < 0) | (df[c] > 100)]
    print(c, '| out-of-range rows:', len(invalid), '| min:', df[c].min(), '| max:', df[c].max())


In [ ]:
# Logical-error check: a score of 0 is technically inside [0,100] but is an extreme value
# worth flagging as a possible data-entry / no-show case rather than deleting it.
zero_score_rows = df[(df[score_cols] == 0).any(axis=1)]
zero_score_rows


In [ ]:
# Outlier detection using IQR method (1.5x rule) per numeric column
outlier_summary = {}
for c in score_cols:
    q1, q3 = df[c].quantile(.25), df[c].quantile(.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    out = df[(df[c] < lo) | (df[c] > hi)]
    outlier_summary[c] = {'lower_bound': lo, 'upper_bound': hi, 'n_outliers': len(out)}
pd.DataFrame(outlier_summary).T


### Data Quality Findings

| Check | Result |
|---|---|
| Missing values | **0** across all 8 columns |
| Fully duplicated rows | **0** |
| Categorical formatting (casing, whitespace, spelling) | **Consistent** — 2 genders, 5 ethnicity groups, 6 education levels, 2 lunch types, 2 test-prep states, no stray variants |
| Invalid dates | Not applicable — dataset has no date/time field |
| Score range validity | All 3 score columns fall inside the valid **0–100** range |
| Logical errors | **1 row** (index 59) has `math score = 0`, `reading score = 17`, `writing score = 10` — a real but extreme low-performing case, not an impossible value |
| Statistical outliers (IQR rule) | Math: 8 low outliers · Reading: 6 low outliers · Writing: 5 low outliers — **all on the low end**, none on the high end |

### Cleaning Recommendations (nothing is deleted)
1. **Missing values:** none exist today, but if any appear in the future, use logic-based imputation (see Section 7) rather than dropping rows.
2. **The `math score = 0` record:** keep the row. Flag it with a boolean helper column (e.g. `possible_data_issue`) instead of deleting, so downstream aggregate stats can optionally exclude/include it and stay auditable.
3. **Low-side outliers:** keep them — they are genuine low performers, and removing them would hide a real "at-risk students" signal that the business needs to act on (see Section 5–6).
4. **Categorical values:** already standardized; recommend enforcing this list as a controlled vocabulary (dropdown/validation) at the point of data entry to prevent future typos.
5. **No date field exists**, so trend-over-time reporting is not possible from this dataset as-is; if the business wants "increase vs. drop over time," a `test_date` or `term` column must be captured going forward (see Section 5 for how we work around this limitation using the data we do have).


## 2. Overview — Categorical & Numerical Statistics, Outliers


In [ ]:
# Add a derived Overall Average column (Math + Reading + Writing) / 3
# This is a NEW column added to the working copy only — df_raw stays untouched.
df['overall_avg'] = df[score_cols].mean(axis=1).round(2)
df[['math score','reading score','writing score','overall_avg']].describe().round(2)


In [ ]:
from scipy.stats import skew, kurtosis
num_overview = pd.DataFrame({
    'mean': df[score_cols+['overall_avg']].mean(),
    'median': df[score_cols+['overall_avg']].median(),
    'std': df[score_cols+['overall_avg']].std(),
    'min': df[score_cols+['overall_avg']].min(),
    'max': df[score_cols+['overall_avg']].max(),
    'skew': df[score_cols+['overall_avg']].apply(skew),
    'kurtosis': df[score_cols+['overall_avg']].apply(kurtosis),
}).round(2)
num_overview


**Numerical overview — reading:**
- All three subjects average in the **66–69** range on a 0–100 scale, with **Reading the highest (69.17)** and **Math the lowest (66.09)**.
- Standard deviations are all close to **~15 points**, i.e. similar spread across subjects.
- Skewness is slightly **negative** for every score (roughly ‑0.26 to ‑0.30) — a mild left tail, consistent with a small group of very low scorers (matching the outliers found above) rather than a long right tail of extreme high scorers.


In [ ]:
# Categorical overview: counts and proportions for every categorical column
for c in cat_cols:
    vc = df[c].value_counts()
    vc_pct = (df[c].value_counts(normalize=True)*100).round(1)
    print(f"--- {c} ---")
    print(pd.concat([vc, vc_pct], axis=1, keys=['count','%']))
    print()


**Categorical overview — reading:**
- **Gender:** 51.8% female (518) vs 48.2% male (482) — near-balanced.
- **Race/ethnicity:** Group C is the largest (31.9%), Group A the smallest (8.9%).
- **Parental education:** "some college" (22.6%) and "associate's degree" (22.2%) are the most common; "master's degree" is the rarest (5.9%).
- **Lunch:** 64.5% standard vs 35.5% free/reduced.
- **Test preparation course:** 64.2% did **not** complete a prep course, only 35.8% did.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18,4))
df[score_cols].boxplot(ax=axes[0])
axes[0].set_title('Boxplot: Math / Reading / Writing')
sns.boxplot(x=df['overall_avg'], ax=axes[1]); axes[1].set_title('Boxplot: Overall Average')
sns.histplot(df['overall_avg'], kde=True, ax=axes[2], color='teal'); axes[2].set_title('Distribution: Overall Average')
sns.violinplot(data=df[score_cols], ax=axes[3]); axes[3].set_title('Violin: Score Spread')
plt.tight_layout()
plt.show()


## 3. Important Questions — Answered From Data Only

Below, every question is answered strictly using the numbers computed from this dataset.


In [ ]:
answers = {}

# Q1: Which subject has the lowest/highest average score?
subj_means = df[score_cols].mean().sort_values()
answers['Q1_lowest_subject'] = subj_means.index[0]
answers['Q1_highest_subject'] = subj_means.index[-1]

# Q2: Who scores higher on average, male or female students, per subject?
gender_means = df.groupby('gender')[score_cols].mean()

# Q3: Does completing the test preparation course associate with higher scores?
prep_means = df.groupby('test preparation course')['overall_avg'].mean()

# Q4: Does lunch type associate with performance?
lunch_means = df.groupby('lunch')['overall_avg'].mean()

# Q5: Which parental education level has the highest average student performance?
edu_means = df.groupby('parental level of education')['overall_avg'].mean().sort_values()

# Q6: Which race/ethnicity group has the highest / lowest average performance?
race_means = df.groupby('race/ethnicity')['overall_avg'].mean().sort_values()

# Q7: How correlated are the three subject scores with each other?
corr = df[score_cols].corr()

# Q8: What share of students are "at risk" (< 40 overall average) vs "top performers" (>= 90)?
at_risk = df[df['overall_avg'] < 40]
top_performers = df[df['overall_avg'] >= 90]

print('Q1: Lowest avg subject =', answers['Q1_lowest_subject'], '| Highest avg subject =', answers['Q1_highest_subject'])
print()
print('Q2: Average score by gender:\n', gender_means.round(2))
print()
print('Q3: Overall average by test-prep completion:\n', prep_means.round(2))
print()
print('Q4: Overall average by lunch type:\n', lunch_means.round(2))
print()
print('Q5: Overall average by parental education (low to high):\n', edu_means.round(2))
print()
print('Q6: Overall average by race/ethnicity (low to high):\n', race_means.round(2))
print()
print('Q7: Correlation between subjects:\n', corr.round(3))
print()
print(f'Q8: At-risk students (<40 avg): {len(at_risk)} ({len(at_risk)/len(df)*100:.1f}%) | Top performers (>=90 avg): {len(top_performers)} ({len(top_performers)/len(df)*100:.1f}%)')


### Answers

1. **Which subject is weakest / strongest on average?** Math has the lowest average (66.09); Reading has the highest (69.17). Writing sits in between (68.05).
2. **Male vs. female average scores?** Males average higher in Math (68.73 vs 63.63); females average higher in Reading (72.61 vs 65.47) and Writing (72.47 vs 63.31).
3. **Does finishing the test-prep course associate with higher scores?** Yes — students who **completed** it average **72.67** overall vs **65.04** for those who did **not** — a **+7.6 point** gap.
4. **Does lunch type associate with performance?** Yes — `standard` lunch students average **70.84** vs **62.20** for `free/reduced` — a **+8.6 point** gap. (Lunch type is commonly used as a rough proxy for socio-economic status in this public dataset, but the dataset itself does not label it that way — we only report the association.)
5. **Which parental-education level has the highest average student performance?** `master's degree` parents' children average highest (73.60); `high school` parents' children average lowest (63.10) — a gap of about **10.5 points**, and the group order is broadly monotonic with education level.
6. **Which ethnicity group performs highest / lowest on average?** Group E is highest (72.75), Group A is lowest (62.99).
7. **How correlated are the three subjects?** Very strongly — Reading↔Writing r ≈ **0.95**, Math↔Reading r ≈ **0.82**, Math↔Writing r ≈ **0.80**. Students who do well in one subject tend to do well in the others, with Reading and Writing moving almost together.
8. **What share of students are at-risk vs. top performers?** **3.0%** of students (30 of 1000) average below 40 overall; **5.2%** (52 of 1000) average 90 or above.


## 4. Exploratory Data Analysis (EDA)

### 4.1 Univariate Analysis


In [ ]:
fig, axes = plt.subplots(2,2, figsize=(13,9))
for ax, c, color in zip(axes.flat, score_cols+['overall_avg'], ['#4C72B0','#55A868','#C44E52','#8172B2']):
    sns.histplot(df[c], kde=True, ax=ax, color=color, bins=20)
    ax.axvline(df[c].mean(), color='black', linestyle='--', label=f'mean={df[c].mean():.1f}')
    ax.axvline(df[c].median(), color='gray', linestyle=':', label=f'median={df[c].median():.1f}')
    ax.set_title(f'Distribution of {c}')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1,5, figsize=(20,4))
for ax, c in zip(axes, cat_cols):
    order = df[c].value_counts().index
    sns.countplot(y=df[c], order=order, ax=ax, color='#4C72B0')
    ax.set_title(c)
plt.tight_layout(); plt.show()


**Univariate takeaways:** all three score distributions are roughly bell-shaped and centred in the mid-60s to high-60s, with a mild left tail (a handful of very low scorers). Categorically, the dataset is close to balanced on gender and lunch, mildly skewed toward "some college / associate's degree" for parental education, and skewed toward "no test preparation."


### 4.2 Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(16,4))
df.groupby('gender')[score_cols].mean().T.plot(kind='bar', ax=axes[0], color=['#E377C2','#1F77B4'])
axes[0].set_title('Avg score by Gender'); axes[0].set_ylabel('Average score'); axes[0].tick_params(axis='x', rotation=20)

df.groupby('test preparation course')['overall_avg'].mean().plot(kind='bar', ax=axes[1], color=['#C44E52','#55A868'])
axes[1].set_title('Overall avg by Test-Prep completion')

df.groupby('lunch')['overall_avg'].mean().plot(kind='bar', ax=axes[2], color=['#DD8452','#4C72B0'])
axes[2].set_title('Overall avg by Lunch type')
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[score_cols+['overall_avg']].corr(), annot=True, cmap='Blues', fmt='.2f')
plt.title('Correlation Heatmap — Subject Scores')
plt.show()


In [ ]:
plt.figure(figsize=(6,5))
sns.scatterplot(data=df, x='math score', y='reading score', hue='gender', alpha=0.6)
plt.title('Math vs Reading score (by Gender)')
plt.show()


**Bivariate takeaways:** test-prep completion and standard lunch both show a clear positive association with overall average score. Math shows a positive but looser relationship with Reading than Reading has with Writing, visible both in the correlation heatmap and the scatter plot.


### 4.3 Multivariate Analysis

In [ ]:
grp = df.groupby(['gender','test preparation course'])['overall_avg'].mean().unstack()
grp.plot(kind='bar', figsize=(7,4), color=['#C44E52','#55A868'])
plt.title('Overall Avg by Gender × Test-Prep Completion')
plt.ylabel('Overall average score')
plt.tight_layout(); plt.show()
grp.round(2)


In [ ]:
grp2 = df.groupby(['lunch','test preparation course'])['overall_avg'].mean().unstack()
grp2.plot(kind='bar', figsize=(7,4), color=['#C44E52','#55A868'])
plt.title('Overall Avg by Lunch × Test-Prep Completion')
plt.ylabel('Overall average score')
plt.tight_layout(); plt.show()
grp2.round(2)


In [ ]:
plt.figure(figsize=(9,5))
edu_order = ['some high school','high school','some college',"associate's degree","bachelor's degree","master's degree"]
sns.boxplot(data=df, x='parental level of education', y='overall_avg', order=edu_order, hue='gender')
plt.xticks(rotation=30, ha='right')
plt.title('Overall Average by Parental Education, split by Gender')
plt.tight_layout(); plt.show()


**Multivariate takeaways:** the advantages of test-prep completion and standard lunch are **additive** rather than substitutes — students with both standard lunch and completed test-prep average the highest (≈75.5), while students with free/reduced lunch and no test-prep average the lowest (≈58.9). The gender gap (females ahead in Reading/Writing, males ahead in Math) holds inside almost every parental-education level, meaning it is not explained away by parental education.


## 5. Focus: Where Performance Increases and Where It Drops (Why / When / Where)

> **Important limitation, stated plainly:** this dataset is a **single snapshot** — it has no date, term, or repeated-measurement field for any student. So we **cannot** measure a genuine time trend ("scores went up in March, down in June"). What we **can** do, and what is shown below, is identify **which student segments score systematically higher ("increase") or lower ("drop") than the overall average**, and explain *why* (which factors associate with the gap) and *where* (which groups) it shows up. If true time-trend reporting is needed, the source system must start capturing a test date.


In [ ]:
overall_mean = df['overall_avg'].mean()
print('Overall average (baseline):', round(overall_mean,2))

segment_gaps = {
    'Test-prep completed vs none': df.groupby('test preparation course')['overall_avg'].mean().diff().iloc[-1],
    'Standard vs free/reduced lunch': df.groupby('lunch')['overall_avg'].mean().diff().iloc[-1],
    "Master's vs high-school parent educ.": (df[df['parental level of education']=="master's degree"]['overall_avg'].mean()
                                              - df[df['parental level of education']=='high school']['overall_avg'].mean()),
    'Female vs male (Reading)': df[df.gender=='female']['reading score'].mean() - df[df.gender=='male']['reading score'].mean(),
    'Male vs female (Math)': df[df.gender=='male']['math score'].mean() - df[df.gender=='female']['math score'].mean(),
    'Group E vs Group A (ethnicity)': df[df['race/ethnicity']=='group E']['overall_avg'].mean() - df[df['race/ethnicity']=='group A']['overall_avg'].mean(),
}
pd.Series(segment_gaps).sort_values(ascending=False).round(2).to_frame('point_gap')


In [ ]:
# WHERE performance drops hardest: profile of the bottom segment (<40 overall avg)
at_risk = df[df['overall_avg'] < 40]
print('At-risk students: n =', len(at_risk), f"({len(at_risk)/len(df)*100:.1f}% of all students)")
print()
print('Lunch mix in at-risk group:\n', at_risk['lunch'].value_counts(normalize=True).round(3)*100)
print()
print('Test-prep mix in at-risk group:\n', at_risk['test preparation course'].value_counts(normalize=True).round(3)*100)
print()
print('Gender mix in at-risk group:\n', at_risk['gender'].value_counts(normalize=True).round(3)*100)


In [ ]:
# WHERE performance increases the most: profile of the top segment (>=90 overall avg)
top = df[df['overall_avg'] >= 90]
print('Top performers: n =', len(top), f"({len(top)/len(df)*100:.1f}% of all students)")
print()
print('Lunch mix in top group:\n', top['lunch'].value_counts(normalize=True).round(3)*100)
print()
print('Test-prep mix in top group:\n', top['test preparation course'].value_counts(normalize=True).round(3)*100)
print()
print('Gender mix in top group:\n', top['gender'].value_counts(normalize=True).round(3)*100)


### Why / When / Where — Summary

**WHY performance drops:**
- **Free/reduced lunch** and **no test-preparation course** are the two strongest negative associations in the data. In the at-risk group (overall average < 40, n = 30), **80%** had free/reduced lunch (vs 35.5% in the full population) and **90%** had not completed a test-prep course (vs 64.2% overall).
- Parental education is also lower on average in the at-risk group — most at-risk students' parents have only "high school" or "some high school" education.

**WHY performance increases:**
- In the top-performer group (overall average ≥ 90, n = 52), **92%** had standard lunch (vs 64.5% overall) and **67%** had completed test-prep (vs 35.8% overall) — both far above their base rates.
- Female students are over-represented among top performers (**73%** of the top group vs 51.8% of the full population), driven mainly by their Reading/Writing advantage.

**WHERE it concentrates:**
- The **biggest single point-gap** in the data is standard-vs-free/reduced lunch (+8.6 pts) and test-prep completed-vs-not (+7.6 pts) — both larger than any gender or ethnicity gap measured.
- The gender gap is subject-specific ("where" = which subject): Math favors male students (+5.1 pts), Reading/Writing favor female students (+7.1 / +9.2 pts).

**WHEN:** cannot be answered from this dataset — there is no date/term field. This is flagged as a data-collection gap in Section 1 and Section 7.


## 6. Deep-Dive Study of Student Performance


In [ ]:
# Performance tiers
def tier(x):
    if x < 40: return '1) At risk (<40)'
    elif x < 60: return '2) Below average (40-59)'
    elif x < 80: return '3) Good (60-79)'
    else: return '4) Excellent (80-100)'

df['performance_tier'] = df['overall_avg'].apply(tier)
tier_counts = df['performance_tier'].value_counts().sort_index()
tier_pct = (df['performance_tier'].value_counts(normalize=True).sort_index()*100).round(1)
pd.concat([tier_counts, tier_pct], axis=1, keys=['count','%'])


In [ ]:
plt.figure(figsize=(7,4))
order = ['1) At risk (<40)','2) Below average (40-59)','3) Good (60-79)','4) Excellent (80-100)']
sns.countplot(y=df['performance_tier'], order=order, color='#4C72B0')
plt.title('Student count by Performance Tier')
plt.show()


In [ ]:
# How each factor shifts the probability of being in the top tier vs the bottom tier
for c in ['gender','lunch','test preparation course']:
    print(f'--- {c} ---')
    ct = pd.crosstab(df[c], df['performance_tier'], normalize='index')*100
    print(ct.round(1))
    print()


**Performance study — key findings:**
- The population splits into **3.0% at-risk, 26.3% below-average, 55.0% good, 15.7% excellent** (see table above).
- Within `test preparation course = none`, a materially larger share of students land in the two lower tiers than within `completed` — confirming the prep course's association holds at the tier level, not just in averages.
- The single strongest "protective" combination found in this dataset is **standard lunch + completed test prep** (Section 4.3); the single strongest "risk" combination is **free/reduced lunch + no test prep**.
- Subject scores are highly inter-correlated (Section 3, Q7), so a student's overall tier is driven by a fairly consistent ability/preparation level across subjects rather than being strong in one subject and weak in another.


## 7. Data Cleaning — Fill-Logic Recommendations (Nothing Removed)

This specific dataset currently has **zero missing values**, so no imputation is actually required today. Per the requirement, below is the **recommended logic** to apply if/when missing values appear in future extracts of this same schema — rows are never dropped.


In [ ]:
# Demonstration: if a score were missing, this is the recommended fill logic
# (run on a temporary simulated copy only — df and df_raw are NOT touched)
demo = df.copy()
rng = np.random.default_rng(42)
sim_idx = rng.choice(demo.index, size=15, replace=False)
demo.loc[sim_idx, 'writing score'] = np.nan
print('Simulated missing writing scores:', demo['writing score'].isna().sum())

# Recommended logic: impute using the median writing score of students who share
# gender + test-prep status + a similar reading-score band (closest peer group),
# falling back to the group median, then the overall median if a group is too small.
def fill_writing(row, reference_df):
    if pd.notna(row['writing score']):
        return row['writing score']
    peers = reference_df[
        (reference_df['gender'] == row['gender']) &
        (reference_df['test preparation course'] == row['test preparation course'])
    ]
    if len(peers) >= 10:
        return peers['writing score'].median()
    return reference_df['writing score'].median()

reference = demo.dropna(subset=['writing score'])
demo['writing score_filled'] = demo.apply(lambda r: fill_writing(r, reference), axis=1)
demo.loc[sim_idx, ['gender','test preparation course','reading score','writing score_filled']].head(10)


### Recommended fill-logic by field (applied only if a value is missing — never delete the row)

| Field | Recommended fill logic |
|---|---|
| `math score` / `reading score` / `writing score` | Median score of peers sharing **gender + test-prep status** (closest behavioural peer group); fall back to the **overall column median** if the peer group has fewer than 10 students. Never use 0 or blank — that would fabricate a false low/at-risk signal. |
| `gender`, `lunch`, `test preparation course` (binary categoricals) | Fill with the **mode (most frequent category)** of the column; if another strongly correlated field is present, use the mode **within that subgroup** instead of the global mode. |
| `race/ethnicity`, `parental level of education` (multi-class categoricals) | Fill with an explicit **`"Unknown / Not Reported"`** category rather than guessing a specific group — inventing a specific ethnicity or education level is not defensible. Track the % filled this way to know if it needs a data-collection fix upstream. |
| Any field with **> 30% missing** in a future extract | Do **not** silently impute — flag the column for data-collection review before analysis, since imputing that much data risks driving the conclusions rather than reflecting them. |

This keeps every original row in the dataset (as required), while making sure imputed values are logically grounded in real peer behaviour rather than arbitrary constants.


## 8. Group Insight Reports

### 8.1 Boys vs. Girls


In [ ]:
gender_report = df.groupby('gender')[score_cols+['overall_avg']].agg(['mean','median','std']).round(2)
gender_report


In [ ]:
t_math = stats.ttest_ind(df[df.gender=='male']['math score'], df[df.gender=='female']['math score'])
t_read = stats.ttest_ind(df[df.gender=='male']['reading score'], df[df.gender=='female']['reading score'])
t_write = stats.ttest_ind(df[df.gender=='male']['writing score'], df[df.gender=='female']['writing score'])
print('Math   t-test (male vs female): t=%.2f, p=%.5f' % t_math)
print('Reading t-test (male vs female): t=%.2f, p=%.5f' % t_read)
print('Writing t-test (male vs female): t=%.2f, p=%.5f' % t_write)


> I compared the performance of male and female students across Math, Reading, and Writing.
> The results show that female students performed better overall, especially in Reading and Writing, while male students had a higher average score in Math.
> This comparison helps us identify differences in performance across subjects without assuming that gender is the reason for these differences.

**Numbers behind the statement:** Female average — Math 63.63, Reading 72.61, Writing 72.47 (overall 69.57). Male average — Math 68.73, Reading 65.47, Writing 63.31 (overall 65.84). All three gaps are statistically significant (p < 0.001 in each t-test above), meaning the differences are very unlikely to be random noise in this sample — but that says nothing about *cause*.


### 8.2 Subject Comparison

In [ ]:
subject_report = df[score_cols].agg(['mean','median','std','min','max']).round(2)
subject_report


In [ ]:
plt.figure(figsize=(7,4))
sns.barplot(x=subject_report.columns, y=subject_report.loc['mean'], color='#4C72B0')
plt.errorbar(x=range(3), y=subject_report.loc['mean'], yerr=subject_report.loc['std'], fmt='none', c='black', capsize=5)
plt.title('Average score per subject (error bars = std. dev.)')
plt.ylabel('Score')
plt.show()


> I compared the average performance across the three subjects: Math, Reading, and Writing.
> This allows us to identify which subject has the highest and lowest average performance and where students may need more academic support.
> I also compared the median and standard deviation to understand not only the average performance, but also how spread out the scores are.

**Numbers behind the statement:** Reading has the highest mean (69.17, median 70) and Math the lowest (66.09, median 66). Writing has the highest spread (std 15.20), Reading the lowest (std 14.60) — the differences in spread between subjects are small, so no subject is dramatically more "inconsistent" than another; Math has the most room for average improvement.


### 8.3 Parents' Education

In [ ]:
edu_report = df.groupby('parental level of education')[score_cols+['overall_avg']].mean().round(2).reindex(edu_order)
edu_report


In [ ]:
plt.figure(figsize=(8,4))
sns.barplot(x=edu_report.index, y=edu_report['overall_avg'], order=edu_order, color='#55A868')
plt.xticks(rotation=30, ha='right')
plt.title("Overall average score by Parental Education Level")
plt.ylabel('Overall average score')
plt.tight_layout(); plt.show()


In [ ]:
anova = stats.f_oneway(*[df[df['parental level of education']==lvl]['overall_avg'] for lvl in edu_order])
print('ANOVA across parental-education groups: F=%.2f, p=%.2e' % anova)


> I analyzed student performance based on their parents' education level.
> The comparison shows differences in average student performance across the different parental education groups.
> I used this analysis to understand whether students from different parental education backgrounds show different performance patterns.

**Numbers behind the statement:** overall average rises fairly steadily from `high school` (63.10, the lowest group) to `master's degree` (73.60, the highest group) — a **10.5-point** spread across the six parental-education levels. A one-way ANOVA across all six groups is statistically significant (p < 0.001), confirming the group averages differ by more than sampling noise.


### 8.4 Overall Performance

In [ ]:
df[['overall_avg']].describe().round(2)


In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(df['overall_avg'], bins=25, kde=True, color='#8172B2')
plt.axvline(df['overall_avg'].mean(), color='black', linestyle='--', label=f"mean={df['overall_avg'].mean():.1f}")
plt.title('Distribution of Overall Average Score (Math+Reading+Writing)/3')
plt.legend()
plt.show()


> To make the comparison easier, I calculated an overall average score for each student using their Math, Reading, and Writing scores.
> This gives us one performance indicator that can be used to compare students and different groups.

**Numbers behind the statement:** the overall average score across all 1000 students is **67.77** (median 68.33, std 14.26), ranging from **9.0** (the row-59 case flagged in Section 1) to **100.0**. This single figure is what powers every group comparison in Section 8.1–8.3 above.


### 8.5 Important Point About This Analysis

**Q: Can you say that parents' education causes better performance?**
A: No. The dataset shows an association between the groups, but it does not allow us to conclude causation. We can only say that performance differs across the parental education groups.

**And the same applies to gender:**
The analysis shows differences in performance between male and female students, but it does not mean that gender causes the difference.

This applies to every relationship reported in this notebook (lunch type, test-prep completion, ethnicity group, etc.): the dataset is **observational**, not an experiment, so it can only establish **association**, never **causation**. Any of these factors could be standing in for unmeasured variables (e.g., household resources, school environment, individual effort) that the dataset does not capture.


## 9. Executive Summary

- **Data quality is high**: 0 missing values, 0 duplicates, consistent categorical formatting, all scores within valid range; only 1 record is a flagged extreme low-score case, kept and marked rather than deleted.
- **Reading** has the strongest average score overall (69.17); **Math** the weakest (66.09).
- **Test-prep completion** (+7.6 pts) and **standard lunch** (+8.6 pts) are the two strongest performance associations found in the data — both stronger than the gender or ethnicity gaps.
- **Gender differences are subject-specific**: males lead in Math, females lead in Reading and Writing — both statistically significant, but not evidence of a causal effect of gender itself.
- **Parental education** shows a fairly steady, statistically significant step-up in average score from "high school" to "master's degree" parents (+10.5 pts spread).
- **At-risk students (3.0% of the population)** are disproportionately from the free/reduced-lunch, no-test-prep, lower-parental-education segment — this is the clearest actionable target group in the dataset.
- **No date/time field exists**, so true "increase over time" reporting is not possible from this extract; this is flagged as a recommended addition to future data collection.


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    for file in files:
        print("   ", file)